# Assignment A1: Predict, then measure
**AIMS Edge ML for Science · Out: after Lab 1 · Due: Friday 27 November, 22:00 · 20% of the course grade**

In Lab 1 you measured your Colab machine's speed, memory bandwidth and overhead. Now you **use those numbers to predict** how other workloads behave,
and only then check. A wrong prediction that you can explain earns more than a lucky one.

| Part | | Weight | What |
|---|---|---|---|
| 1 | 🟥 AI-off | 40% | Two budgeting problems. Type your working into the answer cells. |
| 2 | 🟩 AI-on (code) + 🟥 short answers | 60% | Count, predict and measure two models on Colab, then answer four short questions. |
| Bonus | extra credit | +5 | One short question (+5). An LLM problem and a CNN are unmarked extras. |

**How to submit**
1. Run every cell, so all outputs and plots are visible.
2. **File → Print → Save as PDF.** Check that your plots and printed numbers appear in the PDF.
3. Name the file `A1_Surname_Firstname.pdf`.
4. Email it **by Friday 27 November, 22:00** to **aymantarig@aims.ac.za** and **hassanh@aims.ac.za**, with **sagar@slac.stanford.edu** in CC.
   Subject: `[EdgeML] A1 – Surname Firstname`.

🟥 *AI-off* answers must be your own reasoning, typed by you. Show your working. For 🟩 code, you may use any tools, but you must be able to explain every line.

## 0. Setup

In [ ]:
# --- Colab setup: run once per session (~30 s) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the dataset on your Google Drive between sessions
LAB = "A1_budgets"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

## Part 1 🟥 AI-off: budgets on paper (40%)
Type your working in the answer cells (double-click to edit). Show every step and state your assumptions. Where a problem asks for *your* machine's numbers,
use your Lab 1 table. Use **fp32** unless told otherwise, and 1 MAC = 2 FLOPs.

**P1. Flagging radio interference at a telescope (25%).**

*The setup.* A radio telescope array such as MeerKAT has many dishes. Each dish's receiver digitises the signal from the sky, and the signal processor splits it into
narrow **frequency channels**, like the bands of a graphic equaliser. Each channel then produces a stream of numbers, one per time step, measuring how much power
arrived in that band. Assume **64 dishes**, **4096 channels per dish**, and **1000 samples per second in each channel** (already averaged down from the raw data).

*The problem.* Human-made signals (mobile phones, aircraft radar, satellites) leak into these bands. This **radio-frequency interference (RFI)** shows up as sudden bright
spikes or bursts in a channel's stream. It must be **flagged** (marked as bad) before the data are combined into images of the sky, or it ruins them.

*The model.* Each channel's stream is cut into consecutive, non-overlapping **chunks of 32 samples**, i.e. 32 ms of data from one channel of one dish. Each chunk goes into a
small MLP **32 → 128 → 64 → 2**, whose two outputs score "clean" vs. "interference". Every chunk from every channel of every dish must be checked.

*The budget.* The flags must be ready **within 1 s** of the data arriving, so that the later processing (which averages the data) can drop flagged chunks before they're mixed in.
*(The numbers are simplified, but realistic in scale.)*

- (a) How many chunks per second must be classified? How many FLOP/s does that take in total, and how many GB/s of input data (fp32) flow into the classifier?
- (b) Write $I(B)$, the arithmetic intensity at batch size $B$ (weights read once per batch). Using your **CPU's** ridge point from Lab 1 (`P_cpu ÷ beta_cpu_dram`), what is
  the smallest $B$ at which this workload stops being memory-bound?
- (c) Could your Colab **CPU** keep up? Your **GPU**? Compare the FLOP/s needed with what each *actually achieved* at large batch in Lab 1 (throughput in jets/s × 8512 FLOPs per jet;
  the jet MLP is a similar size, so use it as an estimate), and say how many of each you'd need. Then use Little's law: how many chunks arrive within the 1 s budget,
  and does that leave room for the large batches a GPU needs?

**P2. A wearable classifier (15%).** A device runs a 100k-MAC classifier 10 times per second, in int8. Use the 45 nm energies from Day 1:
int8 multiply 0.2 pJ, int8 add 0.03 pJ, 32-bit on-chip SRAM read 5 pJ, 32-bit DRAM read 640 pJ. Each 32-bit read brings in 4 int8 weights,
and every weight is read once per inference.
- (a) What is the energy per inference with all weights in on-chip SRAM, and with all weights in DRAM?
- (b) What average power does each case draw? How long does a 0.1 Wh battery last on inference alone?
- (c) In one sentence: what does this tell a hardware designer?

*Your answers (P1 and P2):*



## Part 2: predict, then measure (60%)
You'll study two models that differ in one important way:
- **WideMLP** (16 → 512 → 512 → 512 → 5): ~2 MB of weights, which fits in the CPU's cache.
- **HugeMLP** (16 → 4096 → 4096 → 5): ~67 MB of weights, far bigger than any cache.

Everything in Part 2 runs on the Colab **CPU**. Your GPU numbers from Lab 1 come back in Q3.

### 2.1 Setup: your Lab 1 tools and numbers 🟩
Paste your `measure` helper from Lab 1. Then copy the **six numbers from the table at the end of Lab 1** (Section 7). Enter them in base units: FLOP/s, bytes/s and seconds.
For example, 85 GFLOP/s → `85e9`, 12 GB/s → `12e9`, 40 µs → `40e-6`.
- `P_cpu`: CPU peak compute, from the `matmul n=2048` line (Section 5a).
- `beta_cpu_dram`: CPU **main-memory** bandwidth, from the **largest** triad (Section 5b). The arrays are far bigger than any cache, so the data must come from main memory.
- `beta_cpu_cache`: CPU **cache** bandwidth, the **highest** of the smaller triads (Section 5b). Those arrays are small enough to stay in the CPU's cache.
- `t_overhead`: the jet MLP's batch-1 **median** latency (`batch 1, inference_mode`, Section 3). That model does almost no arithmetic, so its time is almost entirely the
  fixed cost PyTorch adds to *every* call. We use it as the overhead any model pays.
- `P_gpu`, `beta_gpu`: GPU fp32 peak and memory bandwidth (Section 5c), used in Q3.

In [ ]:
import time, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

def measure(fn, n_warmup=20, n_iter=200):
    """Your Lab 1 timing helper: per-call times in microseconds."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Your Colab CPU, from your Lab 1 table.
MACHINE = dict(
    P_cpu=None,             # FLOP/s   CPU matmul n=2048            (Lab 1, 5a)
    beta_cpu_dram=None,     # bytes/s  largest CPU triad            (Lab 1, 5b)
    beta_cpu_cache=None,    # bytes/s  highest smaller CPU triad    (Lab 1, 5b)
    t_overhead=None,        # s        jet MLP batch-1 median       (Lab 1, 3)
    P_gpu=None,             # FLOP/s   GPU fp32 matmul              (Lab 1, 5c)
    beta_gpu=None,          # bytes/s  GPU triad                    (Lab 1, 5c)
)
assert None not in MACHINE.values(), "fill in your Lab 1 numbers"

In [ ]:
def mlp(sizes):
    layers = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

MODELS = {"WideMLP": (16, 512, 512, 512, 5), "HugeMLP": (16, 4096, 4096, 5)}

### 2.2 Count (10%) 🟩
Write `counts(sizes)` returning the parameters, MACs per sample, weight bytes (fp32), and I/O bytes per sample (input + output, fp32), as in Lab 1.

In [ ]:
def counts(sizes, dtype_bytes=4):
    """sizes: layer widths, e.g. (16, 512, 512, 512, 5). Returns a dict:
       {"params": ..., "macs": ...,            # per sample, as in Lab 1
        "weight_bytes": ...,                   # params x dtype_bytes
        "io_bytes": ...}                       # (input + output values per sample) x dtype_bytes
    """
    # YOUR CODE HERE
    raise NotImplementedError

CNT = {name: counts(s) for name, s in MODELS.items()}
for name, c in CNT.items():
    print(f"{name}: params {c['params']:,} | MACs {c['macs']:,} | weights {c['weight_bytes']/1e6:.2f} MB | I/O {c['io_bytes']} B per sample")
assert counts((16, 64, 32, 32, 5))["macs"] == 4256

### 2.3 Predict, **before** you measure (15%) 🟩
Use this latency model for one batch of size $B$:
$$t(B) \approx t_\text{overhead} + \max\left(\frac{\text{FLOPs}(B)}{P_\text{peak}},\; \frac{\text{bytes}(B)}{\beta}\right),\qquad \text{bytes}(B) = \text{weights} + B \times \text{I/O}.$$
Choose which bandwidth applies to each model (`beta_cpu_cache` or `beta_cpu_dram`), based on whether its weights fit in cache. Then run the cell **once** and don't edit it again:
the timestamp records when you predicted.

In [ ]:
def predict_latency(c, B, P_cpu, beta, t_overhead):
    """c: a dict from counts(); B: batch size; P_cpu: FLOP/s; beta: bytes/s; t_overhead: seconds. Returns the predicted latency in seconds."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
BETA_CHOICE = {"WideMLP": "?", "HugeMLP": "?"}      # <- "beta_cpu_cache" or "beta_cpu_dram"

In [ ]:
BATCHES = [1, 16, 256, 4096]
PRED = {n: [predict_latency(CNT[n], B, MACHINE["P_cpu"], MACHINE[BETA_CHOICE[n]], MACHINE["t_overhead"]) for B in BATCHES] for n in MODELS}
print("prediction made at", time.strftime("%Y-%m-%d %H:%M:%S"))
for n, ts in PRED.items():
    print(f"{n:8s}", "  ".join(f"B={B}: {t*1e6:10.1f} µs" for B, t in zip(BATCHES, ts)))

### 2.4 Measure (10%) 🟩
For each batch size, time many calls with your `measure` helper and take the **median**. Use `n_iter=50` for B ≤ 256, and `n_iter=5` for B = 4096, whose calls take up to a second each.
Then **repeat the whole sweep 3 times** (`repeats=3`), so you can see how much the result varies from run to run.

In [ ]:
def measure_model(sizes, batches=BATCHES, repeats=3):
    model = mlp(sizes).eval()
    res = np.zeros((repeats, len(batches)))
    # YOUR CODE HERE
    raise NotImplementedError
    return res          # seconds, shape (repeats, len(batches))

MEAS = {n: measure_model(s) for n, s in MODELS.items()}
for n, r in MEAS.items():
    print(f"{n:8s}", "  ".join(f"B={B}: {m*1e6:10.1f} ± {s*1e6:.1f} µs" for B, m, s in zip(BATCHES, r.mean(0), r.std(0))))

fig, ax = plt.subplots(figsize=(7, 4.5))
for n in MODELS:
    l, = ax.loglog(BATCHES, np.array(PRED[n]) * 1e6, "--", label=f"{n} predicted")
    ax.errorbar(BATCHES, MEAS[n].mean(0) * 1e6, MEAS[n].std(0) * 1e6, fmt="o-", c=l.get_color(), label=f"{n} measured")
ax.set(xlabel="batch size", ylabel="latency per batch [µs]", title="prediction vs. measurement"); ax.legend(fontsize=8); ax.grid(alpha=.3, which="both")
plt.show()

### 2.5 Short answers (25%) 🟥
Answer each in one or two lines, using **your** numbers.

**Q1 (10%)** For each model at B = 1 and at B = 4096: is it **overhead-, memory- or compute-bound**? Give the one number from your outputs that shows it.

| | B = 1 | B = 4096 |
|---|---|---|
| WideMLP | | |
| HugeMLP | | |

**Q2 (8%)** Which prediction was furthest from the measurement (model, batch size, and the ratio)? Which term of the latency model (overhead, compute or memory) do you think was wrong, and why?

**Q3 (7%)** Using your Lab 1 **GPU** numbers, from what batch size would **HugeMLP** become compute-bound on the GPU? *(Hint: for this model, $I(B)$ is about $B/2$.)*

*Your answers:*


---
## Bonus
**BQ (+5 points, extra credit).** You must run **WideMLP** at **1 µs** latency. Which *one* change would you make first, and why? Use your Q1 answer. (Think of the Day 1 levers.)

*Your answer:*

The rest of the bonus is unmarked, for those who want more.

**B1. An LLM on a laptop.** A 7-billion-parameter language model generates text one token at a time. Each token needs one pass over **all** the weights, about 2 FLOPs per parameter.
Take β = 100 GB/s and P_peak = 1 TFLOP/s.
- (a) At batch 1 in fp16, what is the arithmetic intensity, and the maximum tokens/s? Which roof limits it?
- (b) Repeat for 4-bit weights.
- (c) If 16 users share the model, so every pass over the weights produces 16 tokens, what changes? Up to how many users does throughput keep growing?
- (d) A laptop has 8 GB of RAM. Which of these configurations can run at all?

**B2. A small CNN.** Extend `counts` to handle `nn.Conv2d` (its MACs depend on the *output* size: `out_channels × out_H × out_W × in_channels × kernel_H × kernel_W`; PyTorch forward hooks are one way to get the output shape),
then predict and measure this model on 1 × 32 × 32 inputs as in Part 2:
`nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.Conv2d(16,32,3,stride=2,padding=1), nn.ReLU(), nn.Conv2d(32,64,3,stride=2,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64,5))`.
Does your latency model work as well for it? What does it leave out?

In [ ]:
# Bonus work (optional)